In [ ]:
# ==========================================
# 08 - XGBOOST BASELINE
# KHỞI TẠO MÔ HÌNH VÀ ĐỌC DEVELOPMENT
# ==========================================

import sys
import json
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR))

import src.imports as lib

from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    hamming_loss
)

RANDOM_SEED = 42
N_SPLITS = 5

lib.np.random.seed(RANDOM_SEED)

split_dir = ROOT_DIR / "data/data-splits"
fold_dir = split_dir / "cv_folds_v1"

output_dir = (
    ROOT_DIR / "experiments/v1_baseline/xgboost/results"
)
output_dir.mkdir(parents=True, exist_ok=True)

df_dev = lib.pd.read_csv(
    split_dir / "development.csv",
    encoding="utf-8-sig"
)

id_col = "ma_hoc_sinh"

choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

score_cols = [
    col for col in df_dev.columns
    if col.startswith(("L6_", "L7_", "L8_", "L9_"))
    and col.endswith(("_hki", "_hkii", "_ca_nam"))
]

X = df_dev[score_cols].copy()
y = df_dev[choice_cols].copy()

assert X.shape == (9000, 96)
assert y.shape == (9000, 9)
assert df_dev[id_col].is_unique
assert X.notna().all().all()
assert y.isin([0, 1]).all().all()
assert y.sum(axis=1).eq(4).all()

# Cấu hình XGBoost Baseline, không tối ưu siêu tham số
xgb_params = {
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
    "tree_method": "hist"
}

print("XGBoost Baseline")
print("Development:", df_dev.shape)
print("X:", X.shape)
print("y:", y.shape)
print("Cấu hình:", xgb_params)

In [ ]:
# ==========================================
# KHỞI TẠO MULTI-OUTPUT XGBOOST
# ==========================================

def create_xgb_model():

    base_model = XGBClassifier(**xgb_params)

    model = MultiOutputClassifier(
        estimator=base_model,
        n_jobs=1
    )

    return model


model = create_xgb_model()

print("Mô hình:", type(model).__name__)
print("Thuật toán:", type(model.estimator).__name__)
print("Số nhãn mục tiêu:", y.shape[1])

In [ ]:
# ==========================================
# LOAD FOLD CỐ ĐỊNH VÀ HUẤN LUYỆN
# ==========================================

fold_data = []

dev_ids = set(df_dev[id_col])

for fold in range(1, N_SPLITS + 1):

    train_ids = lib.pd.read_csv(
        fold_dir / f"fold_{fold}_train_ids.csv",
        encoding="utf-8-sig"
    )[id_col]

    val_ids = lib.pd.read_csv(
        fold_dir / f"fold_{fold}_validation_ids.csv",
        encoding="utf-8-sig"
    )[id_col]

    assert set(train_ids).isdisjoint(set(val_ids))
    assert set(train_ids) | set(val_ids) == dev_ids

    train_idx = lib.np.flatnonzero(
        df_dev[id_col].isin(train_ids).to_numpy()
    )

    val_idx = lib.np.flatnonzero(
        df_dev[id_col].isin(val_ids).to_numpy()
    )

    fold_data.append({
        "fold": fold,
        "train_idx": train_idx,
        "val_idx": val_idx
    })

    print(
        f"Fold {fold}: "
        f"Train={len(train_idx):,} | "
        f"Validation={len(val_idx):,}"
    )


def train_xgboost(X_train, y_train):

    model = create_xgb_model()
    model.fit(X_train, y_train)

    assert len(model.estimators_) == 9

    return model


print("\nPASS: Đã tải 5 fold dùng chung với Random Forest.")

In [ ]:
# ==========================================
# TRAIN XGBOOST + PREDICT PROBABILITIES
# ==========================================

def predict_positive_proba(model, X_val):

    probabilities = []

    for estimator in model.estimators_:

        classes = list(estimator.classes_)

        positive_index = classes.index(1)

        proba = estimator.predict_proba(
            X_val
        )[:, positive_index]

        probabilities.append(proba)

    return lib.np.column_stack(probabilities)


xgb_fold_results = []

for fold_info in fold_data:

    fold = fold_info["fold"]
    train_idx = fold_info["train_idx"]
    val_idx = fold_info["val_idx"]

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # Đảm bảo cả 2 lớp có mặt ở Training
    assert all(
        y_train[col].nunique() == 2
        for col in choice_cols
    ), f"Fold {fold}: Có nhãn chỉ chứa một lớp!"

    model = train_xgboost(
        X_train,
        y_train
    )

    y_proba = predict_positive_proba(
        model,
        X_val
    )

    assert y_proba.shape == (len(X_val), 9)
    assert lib.np.isfinite(y_proba).all()
    assert ((y_proba >= 0) & (y_proba <= 1)).all()

    xgb_fold_results.append({
        "fold": fold,
        "val_idx": val_idx,
        "y_true": y_val.to_numpy(dtype=int),
        "y_proba": y_proba
    })

    print(
        f"Fold {fold}: "
        f"Đã huấn luyện và dự đoán {len(X_val):,} mẫu."
    )

print("\nPASS: Hoàn tất huấn luyện XGBoost trên 5 fold.")

In [ ]:
# ==========================================
# TOP-4 LABEL SELECTION
# ==========================================

def predict_top4(y_proba):

    y_pred = lib.np.zeros_like(
        y_proba,
        dtype=int
    )

    top4_indices = lib.np.argsort(
        -y_proba,
        axis=1,
        kind="stable"
    )[:, :4]

    row_indices = lib.np.arange(
        len(y_proba)
    )[:, None]

    y_pred[row_indices, top4_indices] = 1

    return y_pred


for result in xgb_fold_results:

    result["y_pred"] = predict_top4(
        result["y_proba"]
    )

    assert (
        result["y_pred"].sum(axis=1) == 4
    ).all()

    print(
        f"Fold {result['fold']}: "
        "PASS - Mỗi học sinh được dự đoán đúng 4 môn."
    )

In [ ]:
# ==========================================
# XGBOOST - 5-FOLD EVALUATION
# ==========================================

xgb_metric_records = []
xgb_label_records = []

for result in xgb_fold_results:

    fold = result["fold"]
    y_true = result["y_true"]
    y_pred = result["y_pred"]

    xgb_metric_records.append({
        "Fold": fold,
        "Exact_Match_Accuracy": accuracy_score(
            y_true, y_pred
        ),
        "Macro_F1": f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "Micro_F1": f1_score(
            y_true, y_pred,
            average="micro",
            zero_division=0
        ),
        "Hamming_Loss": hamming_loss(
            y_true, y_pred
        )
    })

    label_scores = f1_score(
        y_true, y_pred,
        average=None,
        zero_division=0
    )

    for label, score in zip(choice_cols, label_scores):
        xgb_label_records.append({
            "Fold": fold,
            "Nhãn": label,
            "F1": score
        })


xgb_metrics = lib.pd.DataFrame(xgb_metric_records)
xgb_label_f1 = lib.pd.DataFrame(xgb_label_records)

metric_cols = [
    "Exact_Match_Accuracy",
    "Macro_F1",
    "Micro_F1",
    "Hamming_Loss"
]

xgb_summary = lib.pd.DataFrame({
    "Model": "XGBoost",
    "Metric": metric_cols,
    "Mean": xgb_metrics[metric_cols].mean().values,
    "Std": xgb_metrics[metric_cols].std().values
})

print("XGBOOST - KẾT QUẢ THEO FOLD")
display(xgb_metrics.round(4))

print("XGBOOST - KẾT QUẢ TỔNG HỢP")
display(xgb_summary.round(4))

print("F1 THEO TỪNG NHÃN")
display(
    xgb_label_f1.pivot(
        index="Nhãn",
        columns="Fold",
        values="F1"
    ).round(4)
)

In [ ]:
# ==========================================
# SAVE XGBOOST BASELINE RESULTS
# ==========================================

# 1. Lưu cấu hình
config = {
    "model": "XGBoost",
    "wrapper": "MultiOutputClassifier",
    "random_seed": RANDOM_SEED,
    "n_splits": N_SPLITS,
    "cv_source": "data/data-splits/cv_folds_v1",
    "training_source": "Development only",
    "n_features": len(score_cols),
    "n_labels": len(choice_cols),
    "prediction_strategy": "Top-4 probabilities",
    "hyperparameters": xgb_params,
    "feature_columns": score_cols,
    "target_columns": choice_cols
}

with open(
    output_dir / "xgb_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(config, f, ensure_ascii=False, indent=4)


# 2. Lưu chỉ số đánh giá
xgb_metrics.to_csv(
    output_dir / "xgb_metrics_5fold.csv",
    index=False,
    encoding="utf-8-sig"
)

xgb_summary.to_csv(
    output_dir / "xgb_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

xgb_label_f1.to_csv(
    output_dir / "xgb_label_f1.csv",
    index=False,
    encoding="utf-8-sig"
)


# 3. Lưu dự đoán của 5 fold
prediction_records = []

for result in xgb_fold_results:

    fold = result["fold"]
    val_idx = result["val_idx"]

    y_true = result["y_true"]
    y_pred = result["y_pred"]
    y_proba = result["y_proba"]

    pred_df = lib.pd.DataFrame({
        id_col: df_dev.iloc[val_idx][id_col].to_numpy(),
        "Fold": fold
    })

    for j, label in enumerate(choice_cols):
        pred_df[f"true_{j+1}"] = y_true[:, j]
        pred_df[f"pred_{j+1}"] = y_pred[:, j]
        pred_df[f"proba_{j+1}"] = y_proba[:, j]

    prediction_records.append(pred_df)


xgb_predictions = lib.pd.concat(
    prediction_records,
    ignore_index=True
)

assert len(xgb_predictions) == 9000
assert xgb_predictions[id_col].is_unique
assert xgb_predictions["Fold"].nunique() == 5

xgb_predictions.to_csv(
    output_dir / "xgb_predictions_5fold.csv",
    index=False,
    encoding="utf-8-sig"
)


# 4. Kiểm tra file
print("=" * 60)
print("XGBOOST BASELINE - SAVED RESULTS")
print("=" * 60)

for file in sorted(output_dir.iterdir()):
    if file.is_file():
        print(file.name)

print(f"\nThư mục: {output_dir.resolve()}")
print("PASS: Đã lưu cấu hình và kết quả XGBoost.")